# Forward-Backward Propagation


## Libraries


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import math
from random import random

## GPU


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")


Device: mps


## Data


## Neural Network


### Multi-Layer Perceptron - Forward Pass, Backward Pass


In [ ]:
class MLP(object):
    def __init__(self, num_inputs=3, hidden_layers=[3, 3], num_outputs=2):
        """
        Constructor for a multi-layer perceptron(MLP).
        Takes a number of inputs, hidden layers layers and a number of outputs
        Args:
            num_inputs: number of inputs
            hidden_layers: a list of ints for the hidden layers
            num_outputs: number of outputs
        """
        self.num_inputs = num_inputs
        self.hidden_layers = hidden_layers
        self.num_outputs = num_outputs

        # create a generic representation of the layers
        layers = [num_inputs] + hidden_layers + [num_outputs]

        # create random connection weights for the layers
        # w - matrix whose dimensions are the number of neurons in the current layer x number of neurons in the next
        # layer for matrix multiplication
        weights = []
        for i in range(len(layers) - 1):
            w = np.random.rand(layers[i], layers[i + 1])
            weights.append(w)
        self.weights = weights

        activations = []
        for k in range(len(layers)):
            a = np.zeros(layers[k])
            activations.append(a)
        self.activations = activations

        # derivatives for each layer
        derivatives = []
        for j in range(len(layers) - 1):
            d = np.zeros((layers[j], layers[j + 1]))
            derivatives.append(d)
        self.derivatives = derivatives

    def forward_propagate(self, inputs):
        """
        Computes forward propagation of the network based on input signals
        Args:
            inputs: input signals 
        Returns:
            activations: output values
        """
        # activation layer for input layer is the input layer itself
        activations = inputs

        # save activations for backpropagation
        self.activations[0] = activations

        # loop through all layers in the network
        for i, w in enumerate(self.weights):
            # calculate net inputs h (matrix multiply)
            # h(xW) in the tutorial
            net_inputs = activations @ w

            # calculate the activation values
            # a = f(h) in the notes
            activations = self._sigmoid(net_inputs)

            # a_3 = s(h_3)
            # h_3 = a_2 * w_2
            self.activations[i + 1] = activations

        # return output layer activation
        return activations

    def back_propagate(self, error, verbose=False):
        """
        Backpropagates an error signal
        Args:
            error: The error to backpropagate 
        Returns:
            error: The final error of the input 
        """
        # dE/dW_i = (y - a_[i + 1] * s'(h_[i + 1])) * a_i
        # s'(h_[i + 1]) = s(h_[i + 1]) * ( 1 - s(h_[i + 1]))
        # s(h_[i + 1]) = a_[i + 1]
        # dE/dW_[i-1] = (y - a_[i + 1]) * s'(h_[i + 1])) * W_i * s'(h_i) * a_[i - 1]
        for i in reversed(range(len(self.derivatives))):
            # get activations for previous layer
            activations = self.activations[i + 1]

            # compute sigmoid derivative
            delta = error * self._sigmoid_derivative(
                activations
            )  # ndarray([0.1, 0.2]) --> ndarray([[0.1, 0.2]])

            # reshape delta to 2d array
            delta_reshaped = delta.reshape(delta.shape[0], -1).T

            if verbose:
                print(f"Layer {i} | Original delta shape: {delta.shape}")
                print(f"Layer {i} | Reshaped delta shape: {delta_reshaped.shape}\n")

            # get activations for current layer
            current_activations = self.activations[
                i
            ]  # ndarray([0.1 0.2]) --> ndarray([[0.1], [0.2]])
            if verbose:
                print(
                    f"Layer {i} | Original current activation dimensions: {current_activations.shape}"
                )

            # reshape activations as to have them as a 2d column matrix
            current_activations = current_activations.reshape(
                current_activations.shape[0], -1
            )
            if verbose:
                print(
                    f"Layer {i} | Reshaped current activations shape: {current_activations.shape}\n"
                )

            # save derivative after applying matrix multiplication
            self.derivatives[i] = current_activations @ delta_reshaped

            # backpropagate next error
            # (y - a_[i + 1]) * s'(h_[i + 1])) * W_i
            error = delta @ self.weights[i].T
            if verbose:
                print(f"W{i} | shape: {self.weights[i].shape}")
                print(f"Layer {i} | Error shape: {error.shape}\n")

            if verbose:
                print(f"derivatives for W{i} | shape: {self.derivatives[i].shape}\n")
                # print(f"derivatives for W{i}: {self.derivatives[i]}\n")

    def gradient_descent(self, learning_rate=1.0, verbose=False):
        """
        Learns by descending the gradient
        Args:
            learning_rate: how fast to learn
        """
        for i in range(len(self.weights)):
            weights = self.weights[i]
            if verbose:
                print(f"Original W{i}: {weights}")
            derivatives = self.derivatives[i]
            weights += derivatives * learning_rate
            if verbose:
                print(f"Updated W{i}: {weights}")

    def train(self, inputs, targets, epochs, learning_rate):
        """
        Trains the model running forward propagation and backward propagation
        Args:
            inputs: X
            targets: Y
            epochs: Num. epochs we want to train the network for. Epoch = #times we want to the feed the whole dataset(training data) into the neural network
            learning_rate: step to apply to gradient descent
        """
        for i in range(epochs):
            sum_errors = 0
            for j, input in enumerate(inputs):
                target = targets[j]
                output = self.forward_propagate(input)
                error = target - output
                self.back_propagate(error)
                self.gradient_descent(learning_rate)
                sum_errors += self._mse(target, output)
            print(f"Epoch: {i + 1} | Error: {sum_errors / len(inputs)}")

        print(f"Training Finished!")
        print(f"--------------------\n")

    def _mse(self, target, output):
        """
        Error(Loss Function) - there are many of these.
        The one chosen for this MLP is the mean square error (MSE) loss function.
        Args:
            target: ground truth
            output: predicted value
        Returns:
            output
        """
        return np.average((target - output) ** 2)

    def _sigmoid_derivative(self, x):
        """
        Derivative of the activation function.
        Args:
            x: input value
        Returns:
            y: output value
        """
        return x * (1.0 - x)

    def _sigmoid(self, x):
        """
        Activation(Non-Linearity Function) - there are many of these.
        The one chosen for this MLP is the sigmoid function
        Args:
            x: input
        Returns:
            y: output
        """
        return 1.0 / (1.0 + np.exp(-x))


## Testing


In [4]:
"""
create an mlp 
input = 2 neurons 
hidden = 1 neuron 
output = 1 neuron 
"""
mlp = MLP(2, [5], 1)

# create dummy data
input = np.array([0.1, 0.2])
target = np.array([0.3])

# forward propagation
output = mlp.forward_propagate(input)

# calculate error
error = target - output

# backward propagation
mlp.back_propagate(error, verbose=True)

# gradient descent
# mlp.gradient_descent(0.1)

Layer 1 | Original delta shape: (1,)
Layer 1 | Reshaped delta shape: (1, 1)

Layer 1 | Original current activation dimensions: (5,)
Layer 1 | Reshaped current activations shape: (5, 1)

W1 | shape: (5, 1)
Layer 1 | Error shape: (5,)

derivatives for W1 | shape: (5, 1)

Layer 0 | Original delta shape: (5,)
Layer 0 | Reshaped delta shape: (1, 5)

Layer 0 | Original current activation dimensions: (2,)
Layer 0 | Reshaped current activations shape: (2, 1)

W0 | shape: (2, 5)
Layer 0 | Error shape: (2,)

derivatives for W0 | shape: (2, 5)



## Multi-Layer Perceptron - Forward Pass, Backward Pass, Training Loop


In [ ]:
class MLP(object):
    def __init__(self, num_inputs=3, hidden_layers=[3, 3], num_outputs=2):
        """
        Constructor for a multi-layer perceptron(MLP).
        Takes a number of inputs, hidden layers layers and a number of outputs
        Args:
            num_inputs: number of inputs
            hidden_layers: a list of ints for the hidden layers
            num_outputs: number of outputs
        """
        self.num_inputs = num_inputs
        self.hidden_layers = hidden_layers
        self.num_outputs = num_outputs

        # create a generic representation of the layers
        layers = [num_inputs] + hidden_layers + [num_outputs]

        # create random connection weights for the layers
        # w - matrix whose dimensions are the number of neurons in the current layer x number of neurons in the next
        # layer for matrix multiplication
        weights = []
        for i in range(len(layers) - 1):
            w = np.random.rand(layers[i], layers[i + 1])
            weights.append(w)
        self.weights = weights

        activations = []
        for k in range(len(layers)):
            a = np.zeros(layers[k])
            activations.append(a)
        self.activations = activations

        # derivatives for each layer
        derivatives = []
        for j in range(len(layers) - 1):
            d = np.zeros((layers[j], layers[j + 1]))
            derivatives.append(d)
        self.derivatives = derivatives

    def forward_propagate(self, inputs):
        """
        Computes forward propagation of the network based on input signals
        Args:
            inputs: input signals 
        Returns:
            activations: output values
        """
        # activation layer for input layer is the input layer itself
        activations = inputs

        # save activations for backpropagation
        self.activations[0] = activations

        # loop through all layers in the network
        for i, w in enumerate(self.weights):
            # calculate net inputs h (matrix multiply)
            # h(xW) in the tutorial
            net_inputs = activations @ w

            # calculate the activation values
            # a = f(h) in the notes
            activations = self._sigmoid(net_inputs)

            # a_3 = s(h_3)
            # h_3 = a_2 * w_2
            self.activations[i + 1] = activations

        # return output layer activation
        return activations

    def back_propagate(self, error):
        """
        Backpropagates an error signal
        Args:
            error: The error to backpropagate 
        Returns:
            error: The final error of the input 
        """
        # dE/dW_i = (y - a_[i + 1] * s'(h_[i + 1])) * a_i
        # s'(h_[i + 1]) = s(h_[i + 1])*( 1 - s(h_[i + 1]))
        # s(h_[i + 1]) = a_[i + 1]
        # dE/dW_[i-1] = (y - a_[i + 1]) * s'(h_[i + 1])) * W_i * s'(h_i) * a_[i - 1]

        for i in reversed(range(len(self.derivatives))):
            # get activations for previous layer
            activations = self.activations[i + 1]

            # compute sigmoid derivative
            delta = error * self._sigmoid_derivative(
                activations
            )  # ndarray([0.1, 0.2]) --> ndarray([[0.1, 0.2]])

            # reshape delta to 2d array
            delta_reshaped = delta.reshape(delta.shape[0], -1).T

            # get activations for current layer
            current_activations = self.activations[
                i
            ]  # ndarray([0.1 0.2]) --> ndarray([[0.1], [0.2]])

            # reshape activations as to have them as a 2d column matrix
            current_activations = current_activations.reshape(
                current_activations.shape[0], -1
            )

            # save derivative after applying matrix multiplication
            self.derivatives[i] = current_activations @ delta_reshaped

            # backpropagate next error
            # (y - a_[i + 1]) * s'(h_[i + 1])) * W_i
            error = delta @ self.weights[i].T

    def gradient_descent(self, learning_rate=1.0, verbose=False):
        """
        Learns by descending the gradient
        Args:
            learning_rate: how fast to learn 
        """
        for i in range(len(self.weights)):
            weights = self.weights[i]
            if verbose:
                print(f"Original W{i}: {weights}")
            derivatives = self.derivatives[i]
            weights += derivatives * learning_rate
            if verbose:
                print(f"Updated W{i}: {weights}")

    def train(self, inputs, targets, epochs, learning_rate):
        """
        Trains model running forward prop and backprop 
        Args:
            inputs: X 
            targets: Y 
            epochs: Num. epochs we want to train the network for. Epoch = #times we want to the feed the whole dataset(training data) into the neural network
            learning_rate: step to apply to gradient descent 
        """
        for i in range(epochs):
            sum_errors = 0
            for input, target in zip(inputs, targets):
                # forward propagation
                output = self.forward_propagate(input)

                # calculate error
                error = target - output

                # backward propagation
                self.back_propagate(error)

                # apply gradient descent
                self.gradient_descent(learning_rate)

                # update total error
                sum_errors += self._mse(target, output)

            # report error
            print(f"Epoch: {i + 1} | Error: {sum_errors / len(inputs)}")

        print()
        print(f"Training Finished!")
        print(f"--------------------\n")

    def _mse(self, target, output):
        """
        Error(Loss Function) - there are many of these. 
        The one chosen for this MLP is the mean square error (MSE) loss function.
        Args:
            target: ground truth 
            output: predicted value 
        Returns:
            output 
        """
        return np.average((target - output) ** 2)

    def _sigmoid_derivative(self, x):
        """
        Derivative of the activation function. 
        Args:
            x: input value 
        Returns:
            y: output value 
        """
        return x * (1.0 - x)

    def _sigmoid(self, x):
        """
        Activation(Non-Linearity Function) - there are many of these. The one chosen for this MLP is the sigmoid function 
        Args:
            x: input 
        Returns:
            y: output 
        """
        return 1.0 / (1.0 + np.exp(-x))


## Testing


In [6]:
inputs = np.array(
    [[random() / 2 for _ in range(2)] for _ in range(1000)]
)  # array([[0.1, 0.2], [0.3, 0.4]])
targets = np.array([[i[0] + i[1]] for i in inputs])  # array([[0.3], [0.7]])

"""
create an mlp 
input = 2 neurons 
hidden = 1 neuron 
output = 1 neuron 
"""
mlp = MLP(2, [5], 1)

mlp.train(inputs, targets, 50, 0.1)

# create dummy data
input = np.array([0.3, 0.1])
target = np.array([0.4])
output = mlp.forward_propagate(input)

print(f"The network believes that {input[0]} + {input[1]} is equal to {output[0]}")

Epoch: 1 | Error: 0.04495812647894926
Epoch: 2 | Error: 0.04017397757102201
Epoch: 3 | Error: 0.03996658231728793
Epoch: 4 | Error: 0.039761143664291225
Epoch: 5 | Error: 0.039539129131798134
Epoch: 6 | Error: 0.03928147174575205
Epoch: 7 | Error: 0.03896721850537909
Epoch: 8 | Error: 0.03857236219688433
Epoch: 9 | Error: 0.03806889339613563
Epoch: 10 | Error: 0.03742423383850144
Epoch: 11 | Error: 0.03660134819347747
Epoch: 12 | Error: 0.035559993444595844
Epoch: 13 | Error: 0.03425971562503075
Epoch: 14 | Error: 0.0326652334092056
Epoch: 15 | Error: 0.030754532380765344
Epoch: 16 | Error: 0.028529015485308257
Epoch: 17 | Error: 0.026023263627852927
Epoch: 18 | Error: 0.023309969150297413
Epoch: 19 | Error: 0.020495223823462833
Epoch: 20 | Error: 0.017702528421284443
Epoch: 21 | Error: 0.015049961991151514
Epoch: 22 | Error: 0.012629453571000244
Epoch: 23 | Error: 0.010495828334692458
Epoch: 24 | Error: 0.008667365242759534
Epoch: 25 | Error: 0.007134141684909949
Epoch: 26 | Error: 0.